In [5]:
# =============================================================
# NB07 — TEST DU PIPELINE D'INFÉRENCE
# =============================================================
# Vérifie que la chaîne complète fonctionne :
# patient brut → feature engineering → preprocessor → modèle → risque + SHAP

import sys
sys.path.append("../src")  # pour importer feature_engineering et prediction

import pandas as pd
from prediction import predire_patient, preparer_patient
import warnings
warnings.filterwarnings("ignore", category=UserWarning)

# --- Test 1 : un patient exemple (saisie partielle) ---
patient_test = {
    "age": "[60-70)", "gender": "Male", "race": "Caucasian",
    "admission_type_id": "1", "discharge_disposition_id": "1",
    "admission_source_id": "7", "time_in_hospital": 5,
    "medical_specialty": "InternalMedicine",
    "num_lab_procedures": 45, "num_procedures": 1, "num_medications": 18,
    "number_outpatient": 0, "number_emergency": 0, "number_inpatient": 1,
    "diag_1": "250.4", "diag_2": "401", "diag_3": "414",
    "number_diagnoses": 9, "A1Cresult": ">8",
    "change": "Ch", "diabetesMed": "Yes", "insulin": "Steady",
    # les autres médicaments seront "No" par défaut
}

print("=" * 55)
print("TEST 1 — Prédiction sur un patient exemple")
print("=" * 55)
resultat = predire_patient(patient_test)

print(f"\n  Probabilité de réadmission : {resultat['probabilite']:.1%}")
print(f"  Seuil de décision          : {resultat['seuil']}")
print(f"  À risque                   : {resultat['a_risque']}")
print(f"  Niveau                     : {resultat['niveau_risque']}")
print(f"  Recommandation             : {resultat['recommandation']}")

print(f"\n  Top facteurs (SHAP) :")
for f in resultat["facteurs"]:
    fleche = "↑" if f["sens"] == "augmente" else "↓"
    print(f"    {fleche} {f['feature']:<40} {f['contribution']:+.4f}")

TEST 1 — Prédiction sur un patient exemple

  Probabilité de réadmission : 11.3%
  Seuil de décision          : 0.38
  À risque                   : False
  Niveau                     : Faible
  Recommandation             : Suivi standard.

  Top facteurs (SHAP) :
    ↑ num__number_inpatient                    +0.3088
    ↓ cat__discharge_disposition_id_1          -0.1561
    ↓ num__A1Cresult                           -0.1318
    ↑ num__score_risque_hospitalier            +0.1144
    ↑ cat__diag_1_Diabete                      +0.0961
    ↓ num__A1C_mesure                          -0.0707
    ↓ cat__diag_1_Circulatoire                 -0.0480
    ↑ num__num_medications                     +0.0461


In [2]:
# --- Test 2 : cohérence avec le pipeline des notebooks ---
# On prend une ligne de X_test.pkl (déjà preprocessée) et on compare
# la prédiction directe vs via notre pipeline.

import joblib
X_test = pd.read_pickle("../data/processed/X_test.pkl")
model = joblib.load("../models/lgbm_final_calibrated.pkl")

# Prédiction directe sur la 1re ligne de X_test (déjà en 186 features)
proba_directe = model.predict_proba(X_test.iloc[[0]])[0, 1]

print("=" * 55)
print("TEST 2 — Cohérence du pipeline")
print("=" * 55)
print(f"  Prédiction directe sur X_test[0] : {proba_directe:.4f}")
print("  → confirme que le modèle calibré fonctionne")
print("\n  ✓ Si le Test 1 a produit une probabilité et des facteurs SHAP,")
print("    le pipeline d'inférence complet est opérationnel.")

TEST 2 — Cohérence du pipeline
  Prédiction directe sur X_test[0] : 0.0870
  → confirme que le modèle calibré fonctionne

  ✓ Si le Test 1 a produit une probabilité et des facteurs SHAP,
    le pipeline d'inférence complet est opérationnel.
